<a href="https://colab.research.google.com/github/kumarsusheel7497-collab/Geospatial-Data-Analysis/blob/main/Geospatial_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving country_metadata.csv to country_metadata.csv


In [2]:
!pip install folium

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap

In [5]:
df = pd.read_csv("country_metadata.csv")
df.head()

,country_code,country_name,iso3,region,income_group,latitude,longitude
0,ABW,Aruba,ABW,Latin America & Caribbean,High income,12.51670,-70.0167
1,AFG,Afghanistan,AFG,"Middle East, North Africa, Afghanistan & Pakistan",Low income,34.52280,69.1761
2,AGO,Angola,AGO,Sub-Saharan Africa,Lower middle income,-8.81155,13.2420
3,ALB,Albania,ALB,Europe & Central Asia,Upper middle income,41.33170,19.8172
4,AND,Andorra,AND,Europe & Central Asia,High income,42.50750,1.5218


In [6]:
print("Shape of dataset:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nColumn names:")
print(df.columns)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Shape of dataset: (217, 7)

First 5 rows:
  country_code country_name iso3  \
0          ABW        Aruba  ABW   
1          AFG  Afghanistan  AFG   
2          AGO       Angola  AGO   
3          ALB      Albania  ALB   
4          AND      Andorra  AND   

                                              region         income_group  \
0                          Latin America & Caribbean          High income   
1  Middle East, North Africa, Afghanistan & Pakistan           Low income   
2                                 Sub-Saharan Africa  Lower middle income   
3                              Europe & Central Asia  Upper middle income   
4                              Europe & Central Asia          High income   

   latitude  longitude  
0  12.51670   -70.0167  
1  34.52280    69.1761  
2  -8.81155    13.2420  
3  41.33170    19.8172  
4  42.50750     1.5218  

Column names:
Index(['country_code', 'country_name', 'iso3', 'region', 'income_group',
       'latitude', 'longitude'],
      d

In [7]:
df = df.drop_duplicates()
print("Duplicate rows removed successfully.")
print("New shape:", df.shape)

Duplicate rows removed successfully.
New shape: (217, 7)


In [8]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
print(df.columns)

Index(['country_code', 'country_name', 'iso3', 'region', 'income_group',
       'latitude', 'longitude'],
      dtype='object')


In [9]:
for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna("Unknown")

In [10]:
if 'sales' in df.columns:
    df['sales'] = pd.to_numeric(df['sales'], errors='coerce')
    df['sales'] = df['sales'].fillna(df['sales'].median())

In [11]:
if 'latitude' in df.columns:
    df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')

if 'longitude' in df.columns:
    df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

In [12]:
if 'latitude' in df.columns and 'longitude' in df.columns:
    df = df.dropna(subset=['latitude', 'longitude'])

In [13]:
if 'region' in df.columns and 'sales' in df.columns:
    region_sales = df.groupby('region')['sales'].sum().sort_values(ascending=False)
    plt.figure(figsize=(10,5))
    sns.barplot(x=region_sales.index, y=region_sales.values)
    plt.title("Sales by Region")
    plt.xticks(rotation=45)
    plt.ylabel("Total Sales")
    plt.show()

In [14]:
location_col = None
if 'city' in df.columns:
    location_col = 'city'
elif 'region' in df.columns:
    location_col = 'region'

if location_col and 'sales' in df.columns:
    top_locations = df.groupby(location_col)['sales'].sum().sort_values(ascending=False).head(10)
    plt.figure(figsize=(10,5))
    sns.barplot(x=top_locations.index, y=top_locations.values)
    plt.title(f"Top 10 {location_col.title()}s by Sales")
    plt.xticks(rotation=45)
    plt.ylabel("Total Sales")
    plt.show()

In [15]:
if 'latitude' in df.columns and 'longitude' in df.columns:
    map_center = [df['latitude'].mean(), df['longitude'].mean()]
    sales_map = folium.Map(location=map_center, zoom_start=5)

    for _, row in df.iterrows():
        popup_text = ""
        if 'sales' in df.columns:
            popup_text += f"Sales: {row['sales']}<br>"
        if 'city' in df.columns:
            popup_text += f"City: {row['city']}"

        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=5,
            popup=popup_text,
            color='blue',
            fill=True,
            fill_opacity=0.6
        ).add_to(sales_map)

    sales_map

In [16]:
if 'latitude' in df.columns and 'longitude' in df.columns:
    heat_data = []
    if 'sales' in df.columns:
        heat_data = df[['latitude', 'longitude', 'sales']].dropna().values.tolist()
    else:
        heat_data = df[['latitude', 'longitude']].dropna().values.tolist()

    heat_map = folium.Map(location=[df['latitude'].mean(), df['longitude'].mean()], zoom_start=5)
    HeatMap(heat_data).add_to(heat_map)
    heat_map

In [17]:

if 'sales' in df.columns:
    if 'presence' in df.columns:
        expansion_candidates = df[(df['sales'] > df['sales'].median()) & (df['presence'] < df['presence'].median())]
        print("High-demand, low-presence locations:")
        print(expansion_candidates.head(10))
    else:
        print("No 'presence' column found. You can still use high-sales locations as potential expansion areas.")

In [18]:
df.to_csv("cleaned_geospatial_data.csv", index=False)
print("Cleaned geospatial dataset saved successfully.")

Cleaned geospatial dataset saved successfully.


In [19]:
from google.colab import files
files.download("cleaned_geospatial_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>